# Episode 13: Web Browsing Agents

LLMs have a knowledge cutoff. The web doesn't. Give your agents browsing ability and they can work with real-time information.

**By the end of this episode, you will:**

- Build a basic web-fetching tool that gives any agent access to live URLs
- Understand how TinyFish orchestrates parallel deep-research pipelines
- Know the trade-offs between simple fetching, managed crawling, and interactive browsing so you can pick the right approach for your use case

## Why Agents Need the Live Web

Every LLM has a training-data horizon. Ask about yesterday's earnings call, today's weather, or a paper published last week and it draws a blank. Web access closes that gap.

The approaches range from dead-simple to enterprise-grade, and each sits at a different point on the complexity-vs-capability curve.

| Approach | Description | Best For |
|----------|-------------|----------|
| **Simple HTTP fetch** | Basic URL fetching with `urllib` | Static pages, APIs |
| **WebSurferAgent** | AG2's built-in browsing agent | Interactive browsing |
| **Browser Use** | AI-driven browser automation | Dynamic pages, forms |
| **Crawl4AI** | LLM-friendly web crawling | Bulk data extraction |
| **TinyFish** | Managed web research service | Deep research pipelines |

We will start with the simplest approach — a plain `urllib` fetch tool — and then look at how TinyFish scales the same idea to production-grade research pipelines.

## Part 1: A Simple Web Fetching Tool

The fastest way to give an agent web access is a function that fetches a URL, strips HTML tags, and hands back the plain text. No external dependencies, no API keys — just Python's built-in `urllib.request`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

### Define the web fetching tool

The tool takes a URL, pulls the raw HTML, strips scripts, styles, and tags, then truncates to 5 000 characters so the response fits comfortably in the LLM's context window.

In [ ]:
import urllib.request
import re


def fetch_webpage(url: str) -> str:
    """Fetch a webpage and return its text content (truncated to 5000 chars)."""
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=10) as response:
            html = response.read().decode("utf-8", errors="ignore")

        # Strip HTML tags to get plain text
        text = re.sub(r"<script[^>]*>.*?</script>", "", html, flags=re.DOTALL)
        text = re.sub(r"<style[^>]*>.*?</style>", "", text, flags=re.DOTALL)
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

        if len(text) > 5000:
            return text[:5000] + "\n\n[Content truncated...]"
        return text
    except Exception as e:
        return f"Error fetching {url}: {str(e)}"

### Create the agent and register the tool

With the tool defined, wiring it up is two lines: create the agent, then register.

In [ ]:
llm_config = LLMConfig({"model": "gpt-4o-mini"})

web_agent = ConversableAgent(
    name="web_agent",
    system_message=(
        "You are a web research assistant. Use the fetch_webpage tool to "
        "retrieve web content when asked about URLs or current information. "
        "Summarize the content clearly and concisely."
    ),
    llm_config=llm_config, human_input_mode="NEVER",
)

In [ ]:
# Register the tool with the agent
web_agent.register_tool(fetch_webpage)

### Take it for a spin

As in earlier episodes, we use a `user` agent to send the message and a specialized agent to respond. The user agent executes the tool when the LLM agent requests it.

In [ ]:
user = ConversableAgent(name="user", human_input_mode="NEVER")

result = user.run(web_agent, message="Fetch https://ag2.ai and tell me what AG2 is about.",
    max_turns=3,
)
result.process()
print(result.summary)

## Part 2: TinyFish — Deep Web Research at Scale

A single-page fetch is fine for quick look-ups, but real research demands more: multiple sources, parallel extraction, cross-referencing, and synthesis. That is what TinyFish provides as a managed service.

### Architecture: The Due Diligence Pipeline

The `due-diligence-with-tinyfish/` example in this repo shows a pattern worth studying. The architecture is elegant — one crawler seeds the data, six specialists work in parallel, a validator checks for contradictions, and a synthesis agent writes the final report.

```
Seed Crawler
    │
    ├── Specialist 1: Financial Analysis
    ├── Specialist 2: Legal Review
    ├── Specialist 3: Market Research
    ├── Specialist 4: Team Assessment
    ├── Specialist 5: Technology Review
    └── Specialist 6: Competitive Landscape
           │
       Validator (checks completeness & contradictions)
           │
       Synthesis Agent (final report)
```

The six specialists run **in parallel**, each deep-crawling a different facet of the target company. This mirrors how a real due-diligence team works — and it finishes in a fraction of the time sequential research would take.

Each specialist receives seed URLs from the crawler, uses TinyFish to deep-crawl relevant pages, extracts structured findings, and returns them to the validator. The validator decides whether any area needs additional digging before the synthesis agent combines everything into a cohesive report.

Skypoint AI reported that this kind of automated research pipeline compresses tasks that used to take hours into **5-10 minutes**. That is the difference between "nice demo" and "production value."

> **See the full implementation in `due-diligence-with-tinyfish/` in this repo.**

## Part 3: WebSurferAgent — AG2's Built-in Browser

Sometimes you need to *interact* with a page, not just read it. AG2's **WebSurferAgent** maintains a browser session across multiple pages — clicking links, filling forms, handling JavaScript-rendered content, and navigating multi-step workflows.

Where the simple fetch tool grabs a snapshot, WebSurferAgent carries on a conversation with the page. That makes it the right choice when the information you need sits behind pagination, login flows, or dynamically loaded content.

For detailed walkthroughs, check the AG2 blog:
- [WebSurferAgent](https://ag2.ai/blog/2025-01-31-WebSurferAgent) — full agent walkthrough
- [Websurfing Tools](https://ag2.ai/blog/2025-01-31-Websurfing-Tools) — lower-level tool approach

## Additional Approaches

### Browser Use

For pages that demand full browser interaction — login forms, single-page apps, JavaScript-heavy sites — **Browser Use** launches a real browser and lets the agent interact with it visually. Think of it as giving your agent a screen and a mouse.

### Crawl4AI

**Crawl4AI** sits between simple fetching and full browsing. It handles JavaScript rendering, structured data extraction, Markdown conversion, and batch crawling — all optimized to produce LLM-friendly output.

## Choosing the Right Approach

There is no single best option here — each approach trades off simplicity against capability. The right choice depends on what you are building.

| Feature | Simple Fetch | TinyFish | WebSurferAgent |
|---------|-------------|----------|----------------|
| **Setup** | Zero deps | API key needed | AG2 built-in |
| **Dynamic pages** | No | Yes | Yes |
| **Multi-page browsing** | No | Yes (crawling) | Yes (interactive) |
| **Parallel research** | Manual | Built-in | Manual |
| **Cost** | Free | Pay per crawl | LLM calls only |
| **Complexity** | Low | Medium | Medium |
| **Best for** | Quick lookups | Deep research | Interactive browsing |

If you just need to pull a page and summarize it, the simple fetch tool is hard to beat. If you are building a research pipeline that needs to cross-reference dozens of sources, TinyFish's parallel crawling will save you serious time. And if the information lives behind interactive flows, WebSurferAgent is the tool for the job.

## Try It Yourself

Modify the `fetch_webpage` tool to also extract the **page title** from the HTML before stripping tags, and return it at the top of the response.

Hint: grab the title with `re.search(r"<title>(.*?)</title>", html, re.IGNORECASE)` before the tag-stripping logic runs.

In [ ]:
# Your code here — modify fetch_webpage to include the page title


## What's Next

In **Episode 14**, you will give your agents a proper web UI — from quick Streamlit prototypes to production-ready AG-UI interfaces.